In [3]:
import kagglehub
import shutil
import os

# Download dataset menggunakan kagglehub
cache_path = kagglehub.dataset_download("rajumavinmar/fake-instagram-profile-dataset")

# Tentukan target folder di /content
dest_path = "/content/fake-instagram-profile-dataset"

# Pindahkan file dari cache ke /content
if os.path.exists(dest_path):
    shutil.rmtree(dest_path)
shutil.copytree(cache_path, dest_path)

print("Dataset berhasil dipindahkan ke:", dest_path)
print("Isi folder:", os.listdir(dest_path))

100%|██████████| 75.1k/75.1k [00:00<00:00, 420kB/s]

Extracting files...
Dataset berhasil dipindahkan ke: /content/fake-instagram-profile-dataset
Isi folder: ['Instagram_fake_profile_dataset.csv']


In [4]:
import pandas as pd
import os

# Membaca dataset CSV dari folder tujuan
csv_path = '/content/fake-instagram-profile-dataset/Instagram_fake_profile_dataset.csv'

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print("=== Informasi Dataset ===")
    print(df.info())
    print("\n=== Statistik Deskriptif ===")
    display(df.describe())
    print("\n=== 5 Baris Pertama Data ===")
    display(df.head())
else:
    print(f"File CSV tidak ditemukan di: {csv_path}")

=== Informasi Dataset ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   profile pic           5000 non-null   int64  
 1   nums/length username  5000 non-null   float64
 2   fullname words        5000 non-null   int64  
 3   nums/length fullname  5000 non-null   float64
 4   name==username        5000 non-null   int64  
 5   description length    5000 non-null   int64  
 6   external URL          5000 non-null   int64  
 7   private               5000 non-null   int64  
 8   #posts                5000 non-null   int64  
 9   #followers            5000 non-null   int64  
 10  #follows              5000 non-null   int64  
 11  fake                  5000 non-null   int64  
dtypes: float64(2), int64(10)
memory usage: 468.9 KB
None

=== Statistik Deskriptif ===


,profile pic,nums/length username,fullname words,nums/length fullname,name==username,description length,external URL,private,#posts,#followers,#follows,fake
count,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5.000000e+03,5000.000000,5000.00000
mean,0.598200,0.166175,1.212800,0.036068,0.007600,20.997200,0.049600,0.227800,103.331400,5.124435e+04,481.381800,0.50000
std,0.490311,0.194733,0.890435,0.106235,0.086855,33.234706,0.217139,0.419455,377.804886,5.963749e+05,761.530132,0.50005
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.00000
25%,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,4.000000e+01,58.000000,0.00000
50%,1.000000,0.089193,1.000000,0.000000,0.000000,1.000000,0.000000,0.000000,10.000000,1.450000e+02,242.000000,0.50000
75%,1.000000,0.300410,2.000000,0.000000,0.000000,31.250000,0.000000,0.000000,81.000000,6.960000e+02,591.000000,1.00000
max,1.000000,0.920000,12.000000,1.000000,1.000000,150.000000,1.000000,1.000000,7389.000000,1.533854e+07,7500.000000,1.00000



=== 5 Baris Pertama Data ===


,profile pic,nums/length username,fullname words,nums/length fullname,name==username,description length,external URL,private,#posts,#followers,#follows,fake
0,1,0.27,0,0.0,0,53,0,0,32,1000,955,0
1,1,0.00,2,0.0,0,44,0,0,286,2740,533,0
2,1,0.10,2,0.0,0,0,0,1,13,159,98,0
3,1,0.00,1,0.0,0,82,0,0,679,414,651,0
4,1,0.00,2,0.0,0,0,0,1,6,151,126,0


In [5]:
print(df['fake'].value_counts())      # harusnya 2500 : 2500
print(df.duplicated().sum())          # jumlah baris duplikat
print(df.groupby('fake').mean().T)    # beda rata-rata fitur antar kelas

fake
0    2500
1    2500
Name: count, dtype: int64
66
fake                              0           1
profile pic                0.982800    0.213600
nums/length username       0.036900    0.295449
fullname words             1.482400    0.943200
nums/length fullname       0.005048    0.067089
name==username             0.000400    0.014800
description length        37.986000    4.008400
external URL               0.099200    0.000000
private                    0.244000    0.211600
#posts                   199.836400    6.826400
#followers            102392.518000   96.186000
#follows                 668.407200  294.356400


In [6]:
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (classification_report, accuracy_score,
                             precision_score, recall_score, f1_score, roc_auc_score)

In [7]:
# 1. Data
df = df.drop_duplicates()
X = df.drop(columns=['fake'])
y = df['fake']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
print('Train:', X_train.shape, '| Test:', X_test.shape)
print('Proporsi kelas train:', y_train.value_counts(normalize=True).round(3).to_dict())
print('Proporsi kelas test :', y_test.value_counts(normalize=True).round(3).to_dict())

Train: (3947, 11) | Test: (987, 11)
Proporsi kelas train: {0: 0.502, 1: 0.498}
Proporsi kelas test : {0: 0.502, 1: 0.498}


In [8]:
# 2. Preprocessing: log1p untuk kolom hitungan yang skewed, lalu StandardScaler
skewed = ['#posts', '#followers', '#follows']
others = [c for c in X.columns if c not in skewed]
preprocess = ColumnTransformer([
    ('log', Pipeline([
        ('log', FunctionTransformer(np.log1p, feature_names_out='one-to-one')),
        ('scale', StandardScaler())]), skewed),
    ('scale', StandardScaler(), others),
])

In [9]:
# 3. Model dan grid hyperparameter
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
models = {
    'Decision Tree': (DecisionTreeClassifier(random_state=42), {
        'clf__max_depth': [3, 5, 8, None],
        'clf__min_samples_leaf': [1, 10, 30]}),
    'Random Forest': (RandomForestClassifier(random_state=42), {
        'clf__n_estimators': [100, 300],
        'clf__max_depth': [None, 10, 20]}),
    'SVM': (SVC(), {
        'clf__C': [0.1, 1, 10],
        'clf__gamma': ['scale', 0.1]}),
}

In [10]:
# 4. Tuning di data train saja, lalu evaluasi sekali di data test
results, fitted = [], {}
for name, (clf, grid) in models.items():
    pipe = Pipeline([('prep', preprocess), ('clf', clf)])
    gs = GridSearchCV(pipe, grid, cv=cv, scoring='f1', n_jobs=-1)
    gs.fit(X_train, y_train)
    best = gs.best_estimator_
    pred = best.predict(X_test)
    if hasattr(best, 'predict_proba'):
        score = best.predict_proba(X_test)[:, 1]
    else:
        score = best.decision_function(X_test)
    fitted[name] = best
    results.append({
        'Model': name,
        'CV F1 (train)': round(gs.best_score_, 4),
        'Test Accuracy': round(accuracy_score(y_test, pred), 4),
        'Test Precision': round(precision_score(y_test, pred), 4),
        'Test Recall': round(recall_score(y_test, pred), 4),
        'Test F1': round(f1_score(y_test, pred), 4),
        'Test ROC-AUC': round(roc_auc_score(y_test, score), 4),
        'Best params': gs.best_params_,
    })

In [11]:
# 5. Tabel perbandingan, diurutkan dari CV F1 (bukan dari data test)
res_df = pd.DataFrame(results).sort_values('CV F1 (train)', ascending=False)
print(res_df.drop(columns='Best params').to_string(index=False))
print()
for _, row in res_df.iterrows():
    print(row['Model'], '->', row['Best params'])

        Model  CV F1 (train)  Test Accuracy  Test Precision  Test Recall  Test F1  Test ROC-AUC
Random Forest         0.9860         0.9828          0.9897       0.9756   0.9826        0.9991
          SVM         0.9787         0.9838          0.9917       0.9756   0.9836        0.9948
Decision Tree         0.9698         0.9777          0.9796       0.9756   0.9776        0.9777

Random Forest -> {'clf__max_depth': None, 'clf__n_estimators': 300}
SVM -> {'clf__C': 10, 'clf__gamma': 'scale'}
Decision Tree -> {'clf__max_depth': None, 'clf__min_samples_leaf': 1}


In [12]:
# 6. Detail model terbaik
best_name = res_df.iloc[0]['Model']
best_model = fitted[best_name]
print('\nModel terbaik (berdasar CV):', best_name)
print(classification_report(y_test, best_model.predict(X_test),
                            target_names=['Asli (0)', 'Palsu (1)']))


Model terbaik (berdasar CV): Random Forest
              precision    recall  f1-score   support

    Asli (0)       0.98      0.99      0.98       495
   Palsu (1)       0.99      0.98      0.98       492

    accuracy                           0.98       987
   macro avg       0.98      0.98      0.98       987
weighted avg       0.98      0.98      0.98       987



In [ ]:
# 7. Feature importance (hanya jika Random Forest)
if best_name == 'Random Forest' or 'Random Forest' in fitted:
    rf = fitted['Random Forest']
    names = rf.named_steps['prep'].get_feature_names_out()
    imp = pd.Series(rf.named_steps['clf'].feature_importances_, index=names)
    print('\nFeature importance (Random Forest):')
    print(imp.sort_values(ascending=False).round(3).to_string())


Feature importance (Random Forest):
scale__profile pic             0.237
log__#followers                0.199
scale__nums/length username    0.176
log__#posts                    0.170
scale__description length      0.106
log__#follows                  0.069
scale__fullname words          0.021
scale__nums/length fullname    0.014
scale__private                 0.006
scale__external URL            0.001
scale__name==username          0.001


In [ ]:
# 8. Simpan model terbaik untuk deployment
joblib.dump(best_model, 'model_terbaik.pkl')
print('\nModel disimpan ke model_terbaik.pkl')


Model disimpan ke model_terbaik.pkl


In [ ]:
from google.colab import files
files.download('model_terbaik.pkl')

import sklearn
print(sklearn.__version__)   # 1.6.1

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

1.6.1


In [ ]:
import pandas as pd
pd.concat([X_train, y_train], axis=1).to_csv('data_latih.csv', index=False)